# 05 — Compute BOP Descriptors

Computes Bond Order Potential (BOP) moment descriptors for all training structures using the BOPfox software.

## Prerequisites / Input files
- `Fe-Mo/Atomsobjects/Fe-Mo-POSCAR-initial-rescaled-AtomsObjects.pkl`
- **BOPfox binary** (not publicly distributed; contact authors for access)
- `bopfoxfeaturizer` Python package (private; available on request)

## Outputs
- `Fe-Mo/Descriptors/CNAV_parallel_Fe-Mo_*.csv` — pre-computed BOP descriptors (included in repo)

## Notes
> ⚠️ This notebook requires the BOPfox software, which is not publicly distributed.

> Pre-computed BOP descriptors for the training set are already provided in `Fe-Mo/Descriptors/` and do not need to be regenerated to run notebooks 07–08.



#  Computation of BOP Features

- input: atoms objects
- output here: all the descriptors comming from BOP (averages, atomic site based, etc)

In [1]:
from Tools.DatasetTools.Commoms import *
os.environ['PATH']+=':'+os.path.join(os.getcwd(),'dependencies/bopfox/src/')
from dependencies.bopfoxfeaturizer.BopFoxFeaturizer.Featurizer import Featurizer, BopfoxFeatures

/home/mariano/.local/micromamba/envs/Test_MLFeMoTCPs/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'bopfox3' from 'ase.calculators' (/home/mariano/.local/micromamba/envs/Test_MLFeMoTCPs/lib/python3.11/site-packages/ase/calculators/__init__.py)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from Tools.DatasetTools import GetBopFeatures as BOPG

## options:

In [ ]:
dataset = 'Fe-Mo' #  'Cr-Co-W'# 'Fe-Mo'
elements = dataset.split('-')
atomsobjectloc = os.path.join(dataset, 'Atomsobjects')
components = dataset.replace('-','')
globalmoments = 16
model_definitions = BOPG.model_definitions
cutoff = 'table'
atoms = ['initial', 'relaxed']
retry = False

In [ ]:
from IPython.display import JSON
import json

In [ ]:
 JSON(json.dumps(BOPG.model_definitions, indent=4, sort_keys=True) )

In [ ]:
target_case = 'EF_nmhcp' # only for correlation plots

In [ ]:
BOPG.AtomsObjects.keys()

In [ ]:
AtomsObjects = {'initial' : load_atoms_objects(dataset,case='POSCAR.initial', scaling='rescaled'),
                'relaxed' : load_atoms_objects(dataset, case='POSCAR.relaxed-all', scaling='noscaled')
               }
BS = load_fully_curated_briefsummary(dataset)

In [ ]:
try:
   from dependencies.bopdftprojections.bopdftprojections.projections import Projections, Canonicalmodel
   has_dftprojections = True
except ImportError:
   has_dftprojections = False
import shutil

In [ ]:
if has_dftprojections:
   P = Projections()
   C = Canonicalmodel()

In [ ]:
for (model, moments, case), result in BOPG.results.items():
    notoutliers = result.index.intersection(BS.index)
    BOPG.results[(model, moments, case)] = result.loc[notoutliers]
    print(result.shape)

In [ ]:
BOPG.results.keys()

In [ ]:
BOPG.results[('0.7projections_0.5os',globalmoments, 'initial')].columns

In [ ]:
BOPG.results[('0.7projections_0.5os', globalmoments, 'initial')].isna().sum(axis=0)#any()

In [ ]:
example = BOPG.results[('0.7projections_0.5os', globalmoments, 'initial')].query('index.str.contains("sigma.FM")')

In [ ]:
example

In [ ]:
np.set_printoptions(precision=2,linewidth=125)

In [ ]:
np.array(example['moments'][0])

# Failed BOP Calculations 

In [ ]:
for (model, moments, case), result in BOPG.results.items():
    print(f'{model} model, {moments} moemnts, {case} atoms:')
    print(result.isna().sum())
    print('-----------------')
    print(result[result.isna().any(axis=1)])
    print('=================')

##  remove bad data

In [ ]:
BOPG.results.keys()

In [ ]:
removenans = [result.dropna(inplace=True) for (model, moments, case), result in BOPG.results.items()]

In [ ]:
for (model, moments, case), result in BOPG.results.items():
    print(f'{model} model, {case} atoms:')
    print(result.isna().sum())
    print(result[result.isna().any(axis=1)])
    print('=================')

# Arrangement and averaging BOP Features 

- expand arrays of atomwise quantities to quantities averages
- calculate shape factors, $b_1 / b_2$ averages (over CP and structure)
- remove features with constant values (null variance)

In [ ]:
BOPG.resultscnav.keys()

In [ ]:
examplemodel = ('0.7projections_0.5os', 16, 'initial')

In [ ]:
BOPG.resultscnav[examplemodel].filter(regex='inf')

In [ ]:
for (model, moments, case), features in BOPG.resultscnav.items():
    print(model, case, features.isnull().any().sum(), features.shape)

In [ ]:
BOPG.resultscnav[('0.7projections_0.5os', 16, 'initial' )].query('index.str.contains("sigma.FM")').filter(regex='normed_moments_.*_0')

# Compare relaxed to guess BOPS

In [ ]:
inimodel = ('0.7projections_0.5os', 16, 'initial' )
rlxmodel = ('0.7projections_0.5os', 16, 'relaxed' )

In [ ]:
BOPG.resultscnav[rlxmodel]

In [ ]:
BOPG.resultscnav[inimodel]

In [ ]:
Moments_RLX = BOPG.resultscnav[rlxmodel].filter(regex='normed_moments')
Moments_INI = BOPG.resultscnav[inimodel].filter(regex='normed_moments')

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
momentstr=r'$\tilde{{\mu}} $'

In [ ]:
for i in range(6):
    fw, fh = plt.rcParams['figure.figsize']
    fig,axs = plt.subplots(1,5, figsize=(fw*5/2, fh))
    
    for CN, ax in zip(['0','CN12', 'CN14', 'CN15', 'CN16'], axs):
        feature = f'normed_moments_{i}_{CN}'
        x = Moments_RLX[feature]
        y = Moments_INI[feature]
        intersection = x.dropna().index.intersection(y.dropna().index)
        r2 = r2_score(y[intersection], x[intersection])
        ax.scatter(x[intersection], y[intersection] , ec='k')
        proj_0_range = [x.min(), x.max()] 
        ftitle = r'$\langle$'+momentstr+fr'$_{{{i}}} \rangle _{{{CN}}} $'
        ax.plot(proj_0_range, proj_0_range, 'k')
        ax.set_title(ftitle+f', $R^2$ = {r2:.2f}')
    axs[0].set_ylabel('guess structure')
    fig.supxlabel('DFT relaxed', fontsize=18)
    fig.tight_layout()

In [ ]:
plt.rc('text', usetex=True)
plt.rc('font', family='serif', size=24)

In [ ]:
intersection = Moments_RLX.dropna().index.intersection(Moments_INI.dropna().index)
X=    np.ravel(Moments_RLX.loc[intersection])
Y=    np.ravel(Moments_INI.loc[intersection])
fig = plt.figure()
ax = fig.add_subplot([0.25, 0.2, 0.65, 0.7])
ax.scatter(X,Y, ec='k')
therange = [X.min(), X.max()]
r2 = r2_score(X,Y)
ax.set_ylabel('all moments,  DFT relaxed')
ax.set_xlabel('all moments, guess structures')
ax.plot(therange,therange, '-k')
ax.set_title(f'moments, $R^2 = ${r2:.5f}', fontsize=18)
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_Moments_rlx-vs-ini.png', dpi=300)


In [ ]:
an_RLX = BOPG.resultscnav[rlxmodel].filter(regex='an')
an_INI = BOPG.resultscnav[inimodel].filter(regex='an')

In [ ]:
anstr=r'$a$'

In [ ]:
for i in range(6):
    fw, fh = plt.rcParams['figure.figsize']
    fig,axs = plt.subplots(1,5, figsize=(fw*5/2, fh))
    
    for CN, ax in zip(['0','CN12', 'CN14', 'CN15', 'CN16'], axs):
        feature = f'an_{i}_{CN}'
        x = an_RLX[feature]
        y = an_INI[feature]
        intersection = x.dropna().index.intersection(y.dropna().index)
    #    reg = np.poly1d(np.polyfit(x[intersection], y[intersection], 1))
    #    ytilde = reg(x)
        r2 = r2_score(y[intersection], x[intersection])
        ax.scatter(x[intersection], y[intersection] , ec='k')
        proj_0_range = [x.min(), x.max()] 
        ftitle = r'$\langle$'+anstr+fr'$_{{{i}}} \rangle _{{{CN}}} $'
        ax.plot(proj_0_range, proj_0_range, 'k')
        ax.set_title(ftitle+f', $R^2$ = {r2:.4f}')
    axs[0].set_ylabel('guess structure')
    fig.supxlabel('DFT relaxed', fontsize=18)
    fig.tight_layout()

In [ ]:
intersection = an_RLX.dropna().index.intersection(an_INI.dropna().index)
X=    np.ravel(an_RLX[intersection])
Y=    np.ravel(an_INI[intersection])
fig = plt.figure()
ax = fig.add_subplot([0.25, 0.2, 0.65, 0.7])
ax.scatter(X,Y, ec='k')
therange = [X.min(), X.max()]
r2 = r2_score(X,Y)
ax.set_ylabel('all moments,  DFT relaxed')
ax.set_xlabel('all moments, guess structures')
ax.plot(therange,therange, '-k')
ax.set_title(fr'$a_{{n}}$, $R^2 = ${r2:.5f}')
fig.savefig('Fe-Mo/graphs/Figure_Fe-Mo_an_rlx-vs-ini.png', dpi=300)


In [ ]:
np.rav

In [ ]:
Y

In [ ]:
X

# Characterization of BOP features 

In [ ]:
for (model, moments, atoms), resultcnav in BOPG.resultscnav.items():
    break

In [ ]:
resultcnav.filter(regex='an')

In [ ]:
def make_corr_matrix(resultscnav, feature_root, model, avn='0'):
    correlation = pd.concat([
        resultscnav.filter(regex=f'{feature_root}_.*_{avn}'), 
                    BOPG.BS[BOPG.target_case]], axis=1).corr().abs()
    return correlation

In [ ]:
def plot_target_correlations(feature, avn='0'):
    fig = plt.figure(figsize = (6,4))
    i = 0
    axs = []
    for (model, moments, atoms), results in BOPG.resultscnav.items():
        if atoms == 'relaxed':
            continue
        i += 1
        correlation_an = make_corr_matrix(results, feature, model, avn)
        correlations = correlation_an[BOPG.target_case][correlation_an.index != BOPG.target_case]
        if i>1:
            axs.append( fig.add_axes([i-1+0.05, 0.15, 0.95,0.7] ) )
            axs[-1].sharey = axs[0]
            axs[-1].set_yticklabels([])
        else:
            axs.append( fig.add_axes([i-1+0.05, 0.15, 0.95,0.7]))
        axs[-1].set_title(model)
        axs[-1].barh(correlations.index,correlations)
    fig.supxlabel(f'correlation to {BOPG.target_case}',x=1.5, y = -0.05)
    return fig, axs

In [ ]:
fig, axs = plot_target_correlations('an')

In [ ]:
fig, axs = plot_target_correlations('an', 'CN12')

In [ ]:
fig, axs = plot_target_correlations('an', 'CN14')

In [ ]:
fig, axs = plot_target_correlations('an', 'CN15')

In [ ]:
fig, axs = plot_target_correlations('an', 'CN16')

In [ ]:
fig, axs = plot_target_correlations('normed_moments')

In [ ]:
fig, axs = plot_target_correlations('normed_moments','CN12')

In [ ]:
fig, axs = plot_target_correlations('normed_moments','CN14')

In [ ]:
BOPG.resultscnav.keys()

In [ ]:
descriptor = 'bn_1_0'
fig, axes = plt.subplots(1, int(len(BOPG.resultscnav)/2), sharex=True,  sharey=True, figsize=(20, 8))
i = 0
for (model, globalmoments, atoms) , resultcnav in BOPG.resultscnav.items():
    if atoms == 'relaxed':
        continue
    i += 1
    axes[i-1].scatter( BOPG.BS[BOPG.target_case][resultcnav[descriptor].index],resultcnav[descriptor] )#.filter(regex='U')
    axes[i-1].set_title(model.replace('0.7projections_0.5os_',''))
fig.supxlabel(BOPG.target_case)
fig.supylabel(descriptor)

In [ ]:
descriptor = 'an_1_0'
df_descriptor = pd.DataFrame([])
for (model, globalmoments, atoms) , resultcnav in BOPG.resultscnav.items():
    if atoms == 'relaxed':
        continue
    i += 1
    add = resultcnav[descriptor]
    add.name = model.replace('0.7projections_0.5os_','')
    df_descriptor = pd.concat([df_descriptor,add], axis = 1)
g = sns.pairplot(df_descriptor )


In [ ]:
descriptor = 'normed_moments_1_CN16'
df_descriptor = pd.DataFrame([])
for (model, globalmoments, atoms) , resultcnav in BOPG.resultscnav.items():
    if atoms == 'relaxed':
        continue
    i += 1
    add = resultcnav[descriptor]
    add.name = model.replace('0.7projections_0.5os_','')
    df_descriptor = pd.concat([df_descriptor,add], axis = 1)
g = sns.pairplot(df_descriptor )


In [ ]:
correlation_normed_moments = make_corr_matrix('normed_moments', 'canonical')
sns.heatmap(correlation_normed_moments)

In [ ]:
correlation_normed_moments = make_corr_matrix('normed_moments', '0.7projections_0.5os')
sns.heatmap(correlation_normed_moments),

In [ ]:
for (model, moments, atoms),results in resultscnav.items():
    if atoms == 'relaxed':
        continue
    fig, ax = plt.subplots()
    for label, an in results.filter(regex='normed_moments_.*_0').items():
        sns.scatterplot(an,BS[target_case], ax = ax, label=label)

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
fig, ax = plt.subplots()
for (model, moments, atoms),results in resultscnav.items():
    if atoms == 'relaxed':
        continue
    scaler = StandardScaler()
    scaled = scaler.fit_transform(results.an_7_0.values.reshape(-1,1))
    ax.hist(scaled, label = model, bins=50, alpha = 0.5)
ax.legend()

In [ ]:
def comparefeatures(featurename, featuresymbol, ax = None, suffix = '_0', which = [None]):
    plotfeature = f'{featurename}{suffix}'
    symbol = fr'$\langle {featuresymbol} \rangle $'
    return_ax_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
        return_ax_fig = True
    ensamble = []
    for model, thisresultcnav in resultscnav.items():
        if model not in which:
            continue
        ensamble.append(thisresultcnav[plotfeature] )
    ensamble = pd.concat(ensamble, axis=0)
    thisrange = [ensamble.min(), ensamble.max()]
    for model, result in resultscnav.items():
        if model not in which:
            continue
        ax.hist(result[plotfeature], density=True, bins=100, alpha=0.81, edgecolor='k', label=model, range=thisrange)
    ax.set_xlabel(symbol)
    ax.legend()
    ax.set_ylabel('Counts')
    if return_ax_fig:
        return fig, ax

In [ ]:
def compare_many_orders(featurename, featuresymbol, norders = 3, which = ['projections']):
    fig, axes = plt.subplots(1, norders, figsize=(7*norders, 5), sharey=True)
    for i, ax in enumerate(axes):
        if 'sos' not in featurename:
            tfeaturename = featurename+f'_{i+1}'
            tfeaturesymbol = featuresymbol+f'_{i+1}'
            comparefeatures(tfeaturename, tfeaturesymbol, which=which,  ax=ax)
            ax.legend([]).remove()
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), bbox_to_anchor=(0.5,1.005))
    fig.subplots_adjust(bottom=0.2) 
    [ax.set_ylabel('') for ax in axes[1:]]
#    fig.tight_layout()
    return fig, axes
    

In [ ]:
resultscnav.keys()

In [ ]:
ensamble = pd.concat([resultscnav[('canonical', 20, 'initial')]['an_1_0'], resultscnav[('0.8projections_0.3os', 20, 'initial')]['an_1_0']], axis=0) #, resultscnav['projections']['an_1_0']

In [ ]:
range=[ensamble.min(axis=0), ensamble.max(axis=0)]

In [ ]:
plt.rc('font', size=22)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(15,5), sharey=True)
hist = axes[0].hist(resultscnav['canonical']['an_1_0'], edgecolor='k', range=range, bins=50 ) #.plot.hist(bins=100)
#hist = axes[1].hist(resultscnav['projections']['an_2_0'], edgecolor='k', range=range, bins=50 ) #.plot.hist(bins=100)
hist = axes[1].hist(resultscnav['projections_os']['an_1_0'], edgecolor='k', range=range, bins=50 ) #.plot.hist(bins=100)
fig.supxlabel(r'$a_1$', )
axes[0].set_ylabel('density counts')
fig.tight_layout()
fig.savefig(f'{dataset}/graphs/{dataset}_a1_distributions.eps',bbox_to_inches='tight')

In [ ]:
comparefeatures ('an_1', 'a_1', which=['projections_sos', 'canonical','0.5projections_os','0.6projections_os', '0.7projections_os','0.8projections_os'])
fig.savefig(f'{dataset}/graphs/{dataset}_Ubond_distributions_1ax.eps')

In [ ]:
fig, axes = comparefeatures('U_bond_atom_list_1_0', 'U_{bond}', which = ['projections', 'canonical','0.7projections_os'], suffix='')
fig.savefig(f'{dataset}/graphs/{dataset}_Ubond_distributions.eps')

In [ ]:
Ubond = {model: result['U_bond_atom_list_1_0'] for model, result in resultscnav.items()}

In [ ]:
outliers = {'Ubond': {model: ubond[ubond < -50] for model, ubond in Ubond.items()} }

In [ ]:
remove_ubond_outliers = [resultscnav[model].drop(index=ubondoutliers.index, inplace=True) for model, ubondoutliers in outliers['Ubond'].items()]

In [ ]:
fig, axes = comparefeatures('U_bond_atom_list_1_0', 'U_{bond}', suffix='', which = ['projections', 'canonical', '0.7projections_os'])
fig.savefig(f'{dataset}/graphs/Ubond_distribution.pdf')
fig.savefig(f'{dataset}/graphs/{dataset}_Ubond_distributions.eps')

In [ ]:
fig, axes = compare_many_orders('normed_moments',r'\mu')
fig.savefig(f'{dataset}/graphs/{dataset}_distributions_of_moments.eps', bbox_to_inches='tight')

In [ ]:
fig, ax = compare_many_orders('sigma', r'\sigma')
fig.savefig(f'{dataset}/graphs/{dataset}_distributions_of_sigma.eps')


In [ ]:
sigma1 = {model: result['sigma_1_0'] for model, result in resultscnav.items()}

In [ ]:
outliers.update({'sigma1': {model: sigma[sigma < -0.50] for model, sigma in sigma1.items()}})

In [ ]:
outliers['sigma1']

In [ ]:
remove_sigma_outliers = [resultscnav[model].drop(index=sigmaoutliers.index, inplace=True) for model, sigmaoutliers in outliers['sigma1'].items()]

In [ ]:
fig, ax  = compare_many_orders('sigma', r'\sigma', which = ['projectinons_os', '0.7projections_os'])
fig.savefig(f'{dataset}/graphs/distribution_of_sigmas.pdf')

In [ ]:
fig, ax  = compare_many_orders('an', 'a')
[tax.set_xlim([-3,1]) for tax in ax]
fig.tight_layout()
fig.savefig(f'{dataset}/graphs/{dataset}_distribution_an.eps', bbox_to_inches='tight')

In [ ]:
an1 = {model: result['an_1_0'] for model, result in resultscnav.items()}

In [ ]:
outliers.update({'an_1': {'canonical': an1['canonical'][an1['canonical'] > -0.2] }})

In [ ]:
remove_sigma_outliers = [resultscnav[model].drop(index=sigmaoutliers.index, inplace=True) for model, sigmaoutliers in outliers['an_1'].items()]

In [ ]:
fig, ax = compare_many_orders('bn', 'b')
fig.savefig(f'{dataset}/graphs/{dataset}_distributions_bn.eps')

In [ ]:
CP = ['0', 'CN12', 'CN13', 'CN14','CN15','CN16']
CPSYMB=['_{0}',  '{CN_{12}}', '{CN_{13}}', '{CN_{14}}','{CN_{15}}','{CN_{16}}']

In [ ]:
featurename = 'normed_moments'
featuresymbol = '\mu'
model = 'canonical'
def compare_cns(featurename, featuresymbol, model, ax = None):
    if ax is None:
        fig, ax = plt.subplots()
    featuretitle = fr'$\langle {featuresymbol} \rangle$'
    for case, symbol in zip(CP,CPSYMB):
        featuremain = fr'{featurename}' 
        featurecol = fr'{featuremain}_{case}'
        ax.set_xlabel(featuretitle)
        thismin = resultscnav[model].filter(regex=featuremain).min().min()
        thismaxn = resultscnav[model].filter(regex=featuremain).max().max()
    #     = fr'${featuresymbol}_{i} $' for i in range(1,4)}
        ax.hist(resultscnav[model][f'{featurecol}'], label=f'${symbol}$',bins=100, edgecolor='k', alpha=0.7, range=[thismin, thismaxn])
    ax.legend()
    ax.set_ylabel('counts')
    ax.set_xlabel(featuretitle)
    return ax

In [ ]:
def compare_cns_manyorders(featurename, featuresymbol, model):
    fig, ax = plt.subplots(1,3, figsize = (21,5), sharey=True)
    for i, tax in enumerate(ax):
        featuremain = fr'{featurename}_{i+1}'
        featuretitle = fr'{featuresymbol}_{i+1}'
        tax = compare_cns(featuremain, featuretitle, model, ax = tax)
        tax.legend().remove()
    handles, labels = ax[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(labels), bbox_to_anchor=(0.5, 1.1))
    ax[0].set_ylabel('counts')
    ax[0].set_ylim([0, 500])

In [ ]:
ax = compare_cns('U_bond_atom_list_1', 'U_{bond}', 'canonical')
ax.set_xlim([-15, 5])

In [ ]:
compare_cns_manyorders('normed_moments', '\mu', 'projections')

In [ ]:
compare_cns_manyorders('normed_moments', '\mu', 'projections_os')

In [ ]:
import Tools.DatasetTools.Tools as tl
plotter = tl.Plotting
selection = resultscnav['canonical'].columns.str.contains('normed_moments')
normedmomentnames = resultscnav['canonical'].columns[selection]
normedmomenttitles = pd.Series({name: re.sub('normed_moments_(.*)_',r'$\\langle \\mu_{\1} \\rangle ', name)+'$' for name in normedmomentnames})
normedmomenttitles = normedmomenttitles.map(lambda s: re.sub(' 0\$','_{0}$', s))
normedmomenttitles = normedmomenttitles.map(lambda s: re.sub(' CN(.*)\$','_{CN_{\\1}}$', s))
fig, ax = plotter.histoff_realfeatures(resultscnav['canonical'],normedmomentnames,normedmomenttitles,(20,20), ncols = 6)
fig, ax = plotter.histoff_realfeatures(resultscnav['projections'],normedmomentnames,normedmomenttitles,(20,20), ncols = 6, fig_ax=(fig, ax))
fig, ax = plotter.histoff_realfeatures(resultscnav['projections_os'],normedmomentnames,normedmomenttitles,(20,20), ncols = 6, fig_ax=(fig, ax))
#fig, ax = plotter.histoff_realfeatures(resultscnav['projections_sos'],normedmomentnames,normedmomenttitles,(20,20), ncols = 6, fig_ax=(fig, ax))

## Shape factors 

In [ ]:
resultscnav['canonical'].filter(regex='sf')

In [ ]:
comparefeatures('sf', 'b_1 / b_2', which = ['projections_os', '0.7projections_os'])

# Correlations

In [ ]:
BS = pd.read_pickle(f'{dataset}/FullyCuratedParsedBriefSummary.pkl')

In [ ]:
ax = sns.scatterplot(data = resultscnav['canonical'][resultscnav['canonical']['bn_2_0']<4], x = 'an_1_0', y ='bn_2_0')
ax = sns.scatterplot(data = resultscnav['projections'][resultscnav['projections']['bn_2_0']<6], x = 'an_1_0', y ='bn_2_0', ax = ax)
ax = sns.scatterplot(data = resultscnav['projections_os'][resultscnav['projections_os']['bn_2_0']<6], x = 'an_1_0', y ='bn_2_0', ax = ax)
ax = sns.scatterplot(data = resultscnav['0.6projections_os'][resultscnav['0.6projections_os']['bn_2_0']<6], x = 'an_1_0', y ='bn_2_0', ax = ax)
ax = sns.scatterplot(data = resultscnav['0.7projections_os'][resultscnav['0.7projections_os']['bn_2_0']<6], x = 'an_1_0', y ='bn_2_0', ax = ax)
ax = sns.scatterplot(data = resultscnav['0.8projections_os'][resultscnav['0.8projections_os']['bn_2_0']<6], x = 'an_1_0', y ='bn_2_0', ax = ax)
ax.legend(['canonical', 'projections', 'projections_os', '0.6projections_os', '0.7projections_os','0.8projections_os'])

##  shape-factors

In [ ]:
BS.index.difference(resultscnav['canonical'].index)

In [ ]:
resultscnav['canonical'].index.difference(BS.index)

In [ ]:
import pdb

In [ ]:
def target_correlation_scatters (featurename, featuresymbol, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    selection={model: BS.index.intersection(result[featurename].index) for model, result in resultscnav.items()}
    [sns.scatterplot(x=resultscnav[model][featurename].loc[selection[model]], y = BS[target_case][selection[model]], ax = ax) for model in models]
    ax.set_ylabel(r'$\Delta E_f$')
    ax.set_xlabel(featuresymbol)
    ax.legend(models)
    return ax

In [ ]:
def correlation_vs_order(feature, featuresymbol, order=[1,2,3,4]):
    fig, axes = plt.subplots(1, len(order),  sharey=True, figsize=(7*4, 5))
    for i, ax in enumerate(axes):
        thisfeaturename = f'{feature}_{order[i]}_0'
        thisfeaturesymbol = f'${featuresymbol}_{order[i]}$'
        ax = target_correlation_scatters(thisfeaturename, thisfeaturesymbol, ax=ax)
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, ncol=len(labels), loc='upper center')

##  The Ubond spread is too big

that might be because the bond integrals are too strong, due to the fact that they are caclulated from dimers, which are not screened. In this close packed structures the interaction is maybe screened by neighbours so the actual interactin is weacker

In [ ]:
ax = target_correlation_scatters('U_bond_atom_list_1_0', r'$\langle U_{bond} \rangle$')
#ax.set_xlim([-50, 0])resultscnav

In [ ]:
correlation_vs_order('an', 'a', order=[1,2,3])

In [ ]:
correlation_vs_order('bn', 'b', order=[1,2,3])

In [ ]:
correlation_vs_order('bn', 'b', order=[6,7,8])

In [ ]:
ax = target_correlation_scatters('sf_0', '$b_1 / b_2$')

In [ ]:
target_correlation_scatters('U_bind', '$U_{bind}$')

In [ ]:
correlation_vs_order('normed_moments', r'\hat{\mu}', order=[1,4,6])

In [ ]:
correlation_vs_order('moments', r'\mu', order=[1,4,6])

In [ ]:
correlation_vs_order('sigma', r'\sigma', order=[1,2,3])

# Compute pearson correlations explicitly 

In [ ]:
model = 'canonical'

In [ ]:
BS[target_case] = BS[target_case].astype(float)

In [ ]:
TargetCorrelations =  {}

In [ ]:
resultscnav['canonical'].astype(float) #['U_bind'].astype(float).corr(BS['E0'])

In [ ]:
for model in models :
    TargetCorrelations[model] = resultscnav[model].corrwith(BS[target_case]).dropna().abs().sort_values(ascending=False)

In [ ]:
models

In [ ]:
normed_moments_corr = resultscnav['projections_os'].filter(regex='normed_moments.*0$').corr()

In [ ]:
sns.heatmap(normed_moments_corr.abs())

In [ ]:
ax = TargetCorrelations['projections_os'][:10].plot(kind='bar')
ax.set_ylabel(r'Corr(x, $\Delta E_f$)')

In [ ]:
ax = TargetCorrelations['0.7projections_os'][:10].plot(kind='bar')
ax.set_ylabel(r'Corr(x, $\Delta E_f$)')

In [ ]:
TargetCorrelations['projections'][:10].plot(kind='bar')

In [ ]:
TargetCorrelations['projections_os'][:10].plot(kind='bar')

In [ ]:
ax = sns.regplot(data = resultscnav['projections_os'], x='bn_7_0', y='bn_8_0', label='projections_os')
#ax = sns.regplot(data = resultscnav['projections_sos'], x='bn_7_0', y='bn_8_0', label='projections_sos')
sns.lineplot(data = resultscnav['projections_os'], x='bn_8_0', y='bn_8_0', ax=ax, linewidth=5, color= 'k')

In [ ]:
ax = sns.scatterplot(data = resultscnav['projections_os'], x='bn_8_0', y='bn_8_CN12' )
ax = sns.scatterplot(data = resultscnav['projections_os'], x='bn_8_0', y='bn_8_CN13', ax=ax )
ax = sns.scatterplot(data = resultscnav['projections_os'], x='bn_8_0', y='bn_8_CN14', ax=ax )
sns.lineplot(data = resultscnav['projections_os'], x='bn_8_0', y='bn_8_0', ax=ax, linewidth=5, color= 'k')

In [ ]:
target_correlation_scatters('Binf_1_0', r'$\langle B_{\infty} \rangle _{CN16}$')

In [ ]:
target_correlation_scatters('Ainf_1_CN12', r'$\langle a_{\infty} \rangle _{CN12}$')

In [ ]:
target_correlation_scatters('bn_8_0', r'$\langle b_{8} \rangle $')

In [ ]:
Features.StrucNames.name='StructureNames'

In [ ]:
df = pd.concat([ Features.StrucNames, resultscnav['projections_os']], axis = 1)

In [ ]:
df.columns

In [ ]:
BS['EF_nmhcp']

In [ ]:
sns.boxplot(data=df, x='StructureNames', y = BS['EF_nmhcp'], hue_order='U_bind' )

In [ ]:
df.columns

# save curated CNAV BOPS

In [ ]:
resultspickle

In [ ]:
curatedfiles = {model: file.replace('parallel', 'curated') for model,file in resultspickle.items()}

In [ ]:
[resultscnav[model].to_pickle(file) for model, file in curatedfiles.items()]

# save features correlated to target for ML 

In [ ]:
preselection = {}

In [ ]:
for model in models:
    selection = TargetCorrelations[model] > 0.2
    preselection[model] = resultscnav[model].loc[:,selection]

In [ ]:
preselection['canonical']

In [ ]:
preselection['projections']

In [ ]:
preselection['projections_os']

In [ ]:
for model in models:
    preselected_location = os.path.join(descriptorlocation, f'preselected_cnav_{model}_BOPds.pkl')
    preselection[model].to_pickle(preselected_location)

# save features with low order of moments

In [ ]:
smallorder = {}

In [ ]:
for model in models:
    smallorder[model]=resultscnav[model].filter(regex='an_[0-6]_|bn_[0-6]|^moments_[0-6]|normed|sigma_[0-6]_')
    smallorder_location = os.path.join(descriptorlocation, f'smallorder_cnav_{model}_BOP.pkl')
    smallorder[model].to_pickle(smallorder_location)

In [ ]:
# doprints = [print(pre) for pre in preselected if 'CN12' in pre]

# Distributions by Magnetism 

In [ ]:
mag = 'FM'

In [ ]:
watch_model = '0.7projections_os'
Mag = extract_mag_from_index(resultscnav[watch_model]) #.index.str.split('.').map(lambda i: i[-1])

In [ ]:
def extract_mag_from_index(features_df : pd.core.frame.DataFrame):
    return features_df.index.str.split('.').map(lambda i: i[-1])

In [ ]:
def get_mag_case_feature_split(feature_df : pd.core.frame.DataFrame, mag:str, Mag = Mag):
    cased_feature : pd.core.frame.DataFrame = feature_df.loc[Mag == mag]
    cased_feature.index = cased_feature.index.str.replace('.NM$|.FM$','')
    return cased_feature
    

In [ ]:
magbop = {watch_model : get_mag_case_feature_split(resultscnav[watch_model], 'FM', Mag = Mag)}
nomagbop = {watch_model: get_mag_case_feature_split(resultscnav[watch_model], 'NM', Mag = Mag)}
Mag2 = extract_mag_from_index(resultscnav_mag[watch_model])
magbop2 = {watch_model : get_mag_case_feature_split(resultscnav_mag[watch_model], 'FM', Mag = Mag2)}
nomagbop2 = {watch_model: get_mag_case_feature_split(resultscnav_mag[watch_model], 'NM', Mag=Mag2)}
nomag_and_mag = nomagbop[watch_model].index.intersection(magbop[watch_model].index)
x = np.linspace(magbop[watch_model].normed_moments_1_0.min(),magbop[watch_model].normed_moments_1_0.max(), 2)
ax = sns.scatterplot(x = magbop[watch_model].normed_moments_1_0, y = nomagbop[watch_model].normed_moments_1_0, label = 'not watching mag')
ax = sns.scatterplot(x = magbop2[watch_model].normed_moments_1_0, y = nomagbop2[watch_model].normed_moments_1_0, label = 'watching mag')
sns.lineplot(x=x,y=x, label = 'x=y', color='k')
ax.set_xlabel('Mag nomred moment 1')
ax.set_ylabel('No Mag normed moment 1')

In [ ]:
(resultscnav[watch_model].normed_moments_1_0 - resultscnav_mag[watch_model].normed_moments_1_0)